# Predicting Electric Vehicle Purchases

This notebook explores a custom machine learning approach for predicting electric vehicle purchase intent.

Rather than importing a pre-built machine learning model, the goal is to implement the core algorithm from scratch using only NumPy and pandas. The model is based on gradient boosting with shallow regression trees, including custom implementations of tree construction, split selection, boosting, validation, and ROC-AUC evaluation.

The objective of this experiment is not necessarily to achieve the highest possible leaderboard score. Instead, it is to explore how machine learning algorithms can be designed and adapted to the structure of a specific dataset, while gaining a deeper understanding of how the underlying methods work.

Some feature-engineering choices and thresholds used in this notebook were identified through previous experiments on the dataset.

In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/playground-series-s6e9/"

train = pd.read_csv(PATH + "train.csv", index_col="id")
test  = pd.read_csv(PATH + "test.csv", index_col="id")

train["Will_Buy_EV"] = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

target = "Will_Buy_EV"

X = train.drop(columns=target)
y = train[target]

In [3]:
def add_features(df):
    df = df.copy()

    df["Env_Subsidy"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["Env_Range"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Env_HomeCharging"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Home_Charging_Possible"].astype(str)
    )

    income_bins = [
        -np.inf,
        67506.5,
        85005.5,
        90200,
        104806,
        108878,
        133966,
        168745.5,
        np.inf
    ]

    df["Income_Band"] = pd.cut(
        df["Annual_Income_USD"],
        bins=income_bins,
        labels=False
    ).astype(str)

    df["Income_EnvSubsidy"] = (
        df["Income_Band"]
        + "_"
        + df["Env_Subsidy"]
    )

    return df

X = add_features(X)
test = add_features(test)

print(X.shape)
print(X.columns[-5:].tolist())

(668665, 18)
['Env_Subsidy', 'Env_Range', 'Env_HomeCharging', 'Income_Band', 'Income_EnvSubsidy']


In [4]:
rng = np.random.default_rng(42)

idx0 = np.where(y.to_numpy() == 0)[0]
idx1 = np.where(y.to_numpy() == 1)[0]

rng.shuffle(idx0)
rng.shuffle(idx1)

n0 = int(len(idx0) * 0.2)
n1 = int(len(idx1) * 0.2)

val_idx = np.concatenate([idx0[:n0], idx1[:n1]])
train_idx = np.concatenate([idx0[n0:], idx1[n1:]])

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [5]:
def auc_score(y, score):
    y = np.asarray(y)
    ranks = pd.Series(score).rank(method="average").to_numpy()

    n1 = y.sum()
    n0 = len(y) - n1

    return (ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

In [6]:
def best_numeric_split(x, r, min_leaf=500):
    x = np.asarray(x, dtype=float)
    r = np.asarray(r, dtype=float)

    order = np.argsort(x)
    x = x[order]
    r = r[order]

    n = len(r)

    s = np.cumsum(r)
    s2 = np.cumsum(r ** 2)

    nl = np.arange(1, n)
    nr = n - nl

    sr = s[-1] - s[:-1]
    s2r = s2[-1] - s2[:-1]

    sse = (s2[:-1] - s[:-1] ** 2 / nl + s2r - sr ** 2 / nr)

    valid = ((x[:-1] != x[1:]) & (nl >= min_leaf) & (nr >= min_leaf))

    sse[~valid] = np.inf

    if np.all(np.isinf(sse)):
        return None

    i = np.argmin(sse)

    base_sse = np.sum((r - r.mean()) ** 2)

    return {
        "type": "numeric",
        "split": (x[i] + x[i + 1]) / 2,
        "gain": (base_sse - sse[i]) / n
    }

In [7]:
def best_categorical_split(x, r, min_leaf=500):
    x = np.asarray(x)
    r = np.asarray(r, dtype=float)

    categories, codes = np.unique(x, return_inverse=True)

    count = np.bincount(codes)
    sums  = np.bincount(codes, weights=r)
    sums2 = np.bincount(codes, weights=r ** 2)

    means = sums / count
    order = np.argsort(means)

    count = count[order]
    sums  = sums[order]
    sums2 = sums2[order]
    categories = categories[order]

    cl  = np.cumsum(count)[:-1]
    sl  = np.cumsum(sums)[:-1]
    s2l = np.cumsum(sums2)[:-1]

    cr  = len(r) - cl
    sr  = sums.sum() - sl
    s2r = sums2.sum() - s2l

    sse = (s2l - sl ** 2 / cl + s2r - sr ** 2 / cr)

    valid = (cl >= min_leaf) & (cr >= min_leaf)
    sse[~valid] = np.inf

    if np.all(np.isinf(sse)):
        return None

    i = np.argmin(sse)

    base_sse = np.sum((r - r.mean()) ** 2)

    return {
        "type": "categorical",
        "split": categories[:i + 1].tolist(),
        "gain": (base_sse - sse[i]) / len(r)
    }

In [8]:
def best_split(X, residuals, min_leaf=500):
    best = None

    for feature in X.columns:
        if pd.api.types.is_numeric_dtype(X[feature]):
            candidate = best_numeric_split(X[feature], residuals, min_leaf)
        else:
            candidate = best_categorical_split(X[feature], residuals, min_leaf)

        if candidate is None:
            continue

        candidate["feature"] = feature

        if best is None or candidate["gain"] > best["gain"]:
            best = candidate

    return best

In [9]:
def split_mask(X, split):
    if split["type"] == "numeric":
        return X[split["feature"]].to_numpy() <= split["split"]

    return np.isin(X[split["feature"]].to_numpy(), split["split"])

In [10]:
def build_tree(X, residuals, depth=0, max_depth=2, min_leaf=500):
    residuals = np.asarray(residuals)

    if depth >= max_depth or len(residuals) < 2 * min_leaf:
        return {
            "type": "leaf",
            "prediction": residuals.mean()
        }

    split = best_split(X, residuals, min_leaf)

    if split is None or split["gain"] <= 0:
        return {
            "type": "leaf",
            "prediction": residuals.mean()
        }

    mask = split_mask(X, split)

    return {
        **split,

        "left": build_tree(
            X.iloc[np.where(mask)[0]],
            residuals[mask],
            depth + 1,
            max_depth,
            min_leaf
        ),

        "right": build_tree(
            X.iloc[np.where(~mask)[0]],
            residuals[~mask],
            depth + 1,
            max_depth,
            min_leaf
        )
    }

In [11]:
def predict_tree(X, tree):
    if tree["type"] == "leaf":
        return np.full(len(X), tree["prediction"])

    mask = split_mask(X, tree)
    pred = np.empty(len(X))

    pred[mask] = predict_tree(
        X.iloc[np.where(mask)[0]],
        tree["left"]
    )

    pred[~mask] = predict_tree(
        X.iloc[np.where(~mask)[0]],
        tree["right"]
    )

    return pred

In [12]:
learning_rate = 0.03
max_trees = 2000
patience = 60

initial = y_train.mean()

pred_train = np.full(len(y_train), initial)
pred_val = np.full(len(y_val), initial)

trees = []

best_auc = 0
best_n = 0
wait = 0

for i in range(max_trees):

    residuals = y_train.to_numpy() - pred_train

    tree = build_tree(X_train, residuals, max_depth=2, min_leaf=500)

    trees.append(tree)

    pred_train += learning_rate * predict_tree(X_train, tree)
    pred_val += learning_rate * predict_tree(X_val, tree)

    auc = auc_score(y_val, pred_val)

    print(
        f"{i + 1:3d} | "
        f"AUC={auc:.6f} | "
        f"root={tree.get('feature', 'leaf')} | "
        f"split={tree.get('split', '')}"
    )

    if auc > best_auc:
        best_auc = auc
        best_n = i + 1
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        break

print()
print("Best AUC:", best_auc)
print("Number of trees:", best_n)

  1 | AUC=0.916428 | root=Income_EnvSubsidy | split=['2_1.0_No', '7_3.0_No', '7_5.0_No', '7_1.0_No', '7_2.0_No', '4_2.0_No', '0_1.0_No', '5_1.0_No', '1_1.0_No', '0_2.0_No', '2_2.0_No', '3_1.0_No', '4_1.0_No', '1_2.0_No', '3_2.0_No', '0_3.0_No', '5_2.0_No', '6_1.0_No', '0_1.0_Yes', '4_3.0_No', '2_3.0_No', '1_3.0_No', '0_4.0_No', '6_2.0_No', '3_3.0_No', '0_5.0_No', '5_3.0_No', '1_4.0_No', '1_1.0_Yes', '2_1.0_Yes', '6_3.0_No', '2_4.0_No', '0_2.0_Yes', '3_4.0_No', '3_1.0_Yes', '4_4.0_No', '5_4.0_No', '5_1.0_Yes', '4_1.0_Yes', '1_5.0_No', '2_5.0_No', '6_4.0_No', '6_1.0_Yes', '1_2.0_Yes', '3_5.0_No', '2_2.0_Yes', '4_5.0_No', '3_2.0_Yes', '4_2.0_Yes', '6_5.0_No', '0_3.0_Yes', '5_5.0_No', '5_2.0_Yes', '7_4.0_No', '6_2.0_Yes', '7_1.0_Yes', '1_3.0_Yes', '0_4.0_Yes', '2_3.0_Yes', '3_3.0_Yes', '7_2.0_Yes', '4_3.0_Yes']
  2 | AUC=0.916428 | root=Income_EnvSubsidy | split=['2_1.0_No', '7_5.0_No', '7_1.0_No', '7_2.0_No', '7_3.0_No', '4_2.0_No', '0_1.0_No', '5_1.0_No', '1_1.0_No', '0_2.0_No', '2_2.0_N

In [13]:
initial = y.mean()

pred = np.full(len(y), initial)
trees = []

for i in range(best_n):

    residuals = y.to_numpy() - pred

    tree = build_tree(X, residuals, max_depth=2, min_leaf=500)

    trees.append(tree)

    pred += learning_rate * predict_tree(X, tree)

    print(f"Tree {i + 1}/{best_n}")

Tree 1/977
Tree 2/977
Tree 3/977
Tree 4/977
Tree 5/977
Tree 6/977
Tree 7/977
Tree 8/977
Tree 9/977
Tree 10/977
Tree 11/977
Tree 12/977
Tree 13/977
Tree 14/977
Tree 15/977
Tree 16/977
Tree 17/977
Tree 18/977
Tree 19/977
Tree 20/977
Tree 21/977
Tree 22/977
Tree 23/977
Tree 24/977
Tree 25/977
Tree 26/977
Tree 27/977
Tree 28/977
Tree 29/977
Tree 30/977
Tree 31/977
Tree 32/977
Tree 33/977
Tree 34/977
Tree 35/977
Tree 36/977
Tree 37/977
Tree 38/977
Tree 39/977
Tree 40/977
Tree 41/977
Tree 42/977
Tree 43/977
Tree 44/977
Tree 45/977
Tree 46/977
Tree 47/977
Tree 48/977
Tree 49/977
Tree 50/977
Tree 51/977
Tree 52/977
Tree 53/977
Tree 54/977
Tree 55/977
Tree 56/977
Tree 57/977
Tree 58/977
Tree 59/977
Tree 60/977
Tree 61/977
Tree 62/977
Tree 63/977
Tree 64/977
Tree 65/977
Tree 66/977
Tree 67/977
Tree 68/977
Tree 69/977
Tree 70/977
Tree 71/977
Tree 72/977
Tree 73/977
Tree 74/977
Tree 75/977
Tree 76/977
Tree 77/977
Tree 78/977
Tree 79/977
Tree 80/977
Tree 81/977
Tree 82/977
Tree 83/977
Tree 84/977
T

In [14]:
prob = np.full(len(test), initial)

for tree in trees:
    prob += learning_rate * predict_tree(test, tree)

print("Min:", prob.min())
print("Max:", prob.max())

Min: -0.1048872465789304
Max: 1.0180546343959718


In [15]:
submission = pd.DataFrame({
    "id": test.index,
    "Will_Buy_EV": prob
})

submission.to_csv("submission.csv", index=False)

submission.head()

,id,Will_Buy_EV
0,668665,0.030007
1,668666,0.019890
2,668667,-0.010462
3,668668,-0.007244
4,668669,0.027173
